
# 1. Programação Orientada a Objetos (POO) e Anatomia de Classes
A Programação Orientada a Objetos é fundamental em *Deep Learning*, visto que os modelos e processamentos de dados operam dentro de estruturas de classes. Uma classe atua como um molde para a criação de objetos, agrupando características (atributos) e ações (métodos).

A anatomia estrutural de uma classe em Python envolve:
* **`class`**: A palavra-reservada que declara a estrutura.
* **Docstrings**: Um texto logo abaixo do nome da classe que serve como cartão de visita ou documentação descritiva da finalidade da classe.
* **`self`**: Parâmetro obrigatório nos métodos internos. Ele representa a própria instância da classe, garantindo que variáveis globais do objeto (atributos) fiquem acessíveis a qualquer método interno.

# 2. Métodos Especiais (Dunder Methods)
O Python possui métodos nativos chamados de *Dunder* (Double Underscore), que começam e terminam com dois sublinhados (`__`). Eles conectam a classe a funções nativas da linguagem:
* **`__init__`**: É o método construtor. O interpretador o chama automaticamente no momento em que a classe é instanciada, servindo para inicializar os atributos básicos.
* **`__str__`**: Formata e retorna o texto exibido quando a função `print()` é chamada sobre o objeto.
* **`__len__`**: Conecta o objeto à função `len()` do Python, retornando o tamanho estrutural ou quantidade de elementos do objeto.


In [1]:
class Vetor:
    """
    Docstring: Classe que representa um vetor matemático 2D.
    """
    
    def __init__(self, x, y):
        # A inicialização define os atributos globais do objeto usando o 'self'
        self.x = x
        self.y = y
        
    def magnitude(self):
        # O parâmetro 'self' permite acessar os valores salvos no construtor
        return (self.x**2 + self.y**2) ** 0.5
        
    def __len__(self):
        # Define que o tamanho estrutural deste objeto sempre retorna 2
        return 2
        
    def __str__(self):
        # Define a representação textual do objeto
        return f"Vetor 2D: ({self.x}, {self.y})"

# Instanciação e utilização do objeto
v2d = Vetor(3, 4)
print(v2d)               # Ocorre a chamada implícita do método __str__
print(v2d.magnitude())   # Retorna 5.0
print(len(v2d))          # Ocorre a chamada implícita do método __len__

Vetor 2D: (3, 4)
5.0
2



# 3. Herança
A herança é um pilar da POO que permite o reaproveitamento de código. Uma classe "filha" recebe (herda) os métodos e atributos de uma classe "pai". 

Esse conceito é amplamente aplicado em *Deep Learning*. Por exemplo, o algoritmo de *backpropagation* é idêntico para a maioria dos modelos neurais. Assim, implementa-se essa operação complexa apenas uma vez em uma classe base, e todos os novos modelos apenas herdam essa classe, evitando a reescrita de código.

Na sintaxe do Python, a classe pai é informada entre parênteses ao lado do nome da classe filha.


In [2]:
# A classe Vetor3D herda as propriedades da classe Vetor
class Vetor3D(Vetor):
    """
    Vetor 3D que herda as características da classe base Vetor.
    """
    def __init__(self, x, y, z):
        # O construtor da classe pai é chamado para reaproveitar a lógica de X e Y
        Vetor.__init__(self, x, y) 
        self.z = z 
        
    def magnitude(self):
        # Sobrescrita (override) do método pai para incluir a dimensão Z
        return (self.x**2 + self.y**2 + self.z**2) ** 0.5
        
    def __len__(self):
        # Sobrescrita para refletir as 3 dimensões
        return 3

# Teste da herança
v3d = Vetor3D(3, 4, 12)
print("Magnitude em 3D:", v3d.magnitude()) # Retorna 13.0

Magnitude em 3D: 13.0



# 4. Aplicação: Custom Dataset no PyTorch
Para o processamento de dados no *framework* PyTorch, cria-se uma classe que herda as funcionalidades essenciais da classe nativa `Dataset`. É obrigatório sobrescrever três métodos para que o carregador de dados funcione corretamente:
1. **`__init__`**: Processa o arquivo ou fonte de dados, separando os atributos preditivos (X) da variável alvo (Y) e os convertendo em tensores.
2. **`__len__`**: Retorna a contagem total de instâncias registradas, para definir a escala das épocas de treinamento.
3. **`__getitem__`**: Recebe um `index` e retorna o par de amostra e rótulo correspondente. O PyTorch usa isso para alimentar os lotes iterativamente, aplicando funções de embaralhamento (*shuffle*) automaticamente.

# 5. O Papel dos Dados Sintéticos
Em ambientes de experimentação, muitas vezes não se tem o arquivo real em mãos no momento de construir a classe. A solução é criar um **texto ou conjunto de dados sintético**. 

Dados sintéticos são dados artificiais e aleatórios criados por código (usando `numpy` ou o módulo nativo de arquivos do Python) para simular o comportamento de um banco de dados real. Gerando um CSV temporário com linhas aleatórias apenas para testar se a classe `Dataset` carrega a estrutura corretamente.


In [3]:
%pip install numpy pandas torch

Note: you may need to restart the kernel to use updated packages.


In [4]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset
import os

# Cria-se um arquivo CSV sintético simulando 100 linhas e 5 colunas
# As 4 primeiras colunas são atributos, a última é o rótulo, o rotulo é chamado de "Alvo" que representa a variável dependente que queremos prever
dados_sinteticos = np.random.rand(100, 5)
df_sintetico = pd.DataFrame(dados_sinteticos, columns=['A', 'B', 'C', 'D', 'Alvo'])
df_sintetico.to_csv("dataset_sintetico_temp.csv", index=False)

class CustomDatasetPyTorch(Dataset):
    def __init__(self, caminho_csv):
        # A classe carrega o dado sintético que acabou de ser criado
        df = pd.read_csv(caminho_csv)
        
        # O fatiamento isola o X (todas colunas exceto a última) e o Y (última)
        self.X = torch.tensor(df.iloc[:, :-1].values, dtype=torch.float32)
        self.Y = torch.tensor(df.iloc[:, -1].values, dtype=torch.float32)
        
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, index):
        # Retorna a amostra sob demanda
        return self.X[index], self.Y[index]

# Instanciação com o dado sintético
meu_dataset = CustomDatasetPyTorch("dataset_sintetico_temp.csv")

# Extração do primeiro elemento usando a chamada implícita de __getitem__
amostra_x, amostra_y = meu_dataset[0]
print("Tamanho do Dataset Sintético:", len(meu_dataset))
print("Tensor X (Amostra 0):", amostra_x)

# Limpeza do arquivo sintético temporário
os.remove("dataset_sintetico_temp.csv")

Tamanho do Dataset Sintético: 100
Tensor X (Amostra 0): tensor([0.0084, 0.9935, 0.1357, 0.9794])



# 6. Desafio: Classe Perceptron
O desafio proposto na aula consiste em implementar do zero uma classe para o algoritmo *Perceptron*. Um classificador estruturado sob a ótica de orientação a objetos na área de aprendizado de máquina demanda, no mínimo, as seguintes funções:
* **`fit()`**: Recebe as matrizes X (treino) e Y (marcações) e ajusta matematicamente os pesos internos da rede.
* **`predict()`**: Avalia uma nova matriz X (teste) e devolve as predições.
* **`__len__`**: Método *dunder* que devolve a quantidade de parâmetros ou a estrutura de pesos que a rede aprendeu.


In [5]:
class Perceptron:
    """
    Classificador Perceptron de camada única.
    """
    def __init__(self, learning_rate=0.01, epochs=100):
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.weights = None
        self.bias = None
        
    def fit(self, X, Y):
        # Define os pesos baseados na quantidade de características (colunas)
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features)
        self.bias = 0
        
        # Laço de treinamento
        for _ in range(self.epochs):
            for i, x_i in enumerate(X):
                linear_output = np.dot(x_i, self.weights) + self.bias
                y_predicted = 1 if linear_output >= 0 else 0
                
                # Regra de atualização
                error = Y[i] - y_predicted
                self.weights += self.learning_rate * error * x_i
                self.bias += self.learning_rate * error

    def predict(self, X):
        # Utiliza a rede treinada para novas amostras
        linear_output = np.dot(X, self.weights) + self.bias
        return np.where(linear_output >= 0, 1, 0)
        
    def __len__(self):
        # O tamanho da rede representa a sua quantidade de parâmetros
        return len(self.weights) if self.weights is not None else 0

# Simulação Simples (Operador Lógico OR corrigido com dados base)
X_train = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y_train = np.array([0, 1, 1, 1])

perceptron_model = Perceptron(epochs=10)
perceptron_model.fit(X_train, y_train)

print("Predições da rede perceptron:", perceptron_model.predict(X_train))
print("Tamanho estrutural (parâmetros da rede):", len(perceptron_model))

Predições da rede perceptron: [0 1 1 1]
Tamanho estrutural (parâmetros da rede): 2
